In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# §6.22 The Variational Method and Variational Monte Carlo

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.22",
    title="The Variational Method and Variational Monte Carlo",
    blurb="A good guess, and a guarantee. When a system is too far from anything "
    "solvable for perturbation theory, we can still bound its ground-state energy "
    "from above with nothing but a trial wavefunction — and the better the guess, the "
    "tighter the bound. We optimize simple trials for the oscillator and hydrogen, "
    "screen the nucleus to solve helium, and then let Monte Carlo sample the "
    "wavefunction directly, arriving at the method that powers much of modern "
    "many-body physics.",
    difficulty="advanced",
    estimate="180–220 min",
)

## Notebook overview

The last notebook gave us perturbation theory, which works when a problem is *close* to a solvable one.
This notebook gives us its complement — the **variational method** — which needs no nearby solvable
Hamiltonian at all, only a plausible **guess** at the shape of the ground-state wavefunction. In return
it hands us something rare in the business of approximation: a **rigorous bound**. For *any* normalized
trial state, the energy expectation $\langle\psi|H|\psi\rangle$ is an **upper bound** on the true
ground-state energy $E_0$, with equality only when $\psi$ *is* the ground state. So we pick a family of
trial wavefunctions with adjustable parameters, minimize the energy over them, and know that whatever we
get can never dip below the truth — *lower is always better*.

We build it in three stages. First the **warm-ups**: a Gaussian trial for the harmonic oscillator and an
exponential for hydrogen, where the trial family happens to *contain* the exact ground state, so the
bound is saturated and we recover $\tfrac12$ and $-\tfrac12\,$Ha exactly — a check that the machinery is
sound. Then the **flagship**: the **helium atom**, where two electrons repel through a $1/r_{12}$ term
that is neither small nor separable and that defeats low-order perturbation theory. A product of two
hydrogenic orbitals with an *effective* nuclear charge $Z$ as the single variational parameter captures
the essential physics — each electron partially **screens** the nucleus from the other, so the charge
each sees is *less* than $Z=2$ — and minimizing gives $Z_{\text{eff}}=27/16\approx1.69$ and an energy of
$-2.85\,$Ha, remarkably close to the experimental $-2.90$ and far better than the unscreened guess. One
parameter buys the screening that perturbation theory missed.

Finally, when the best trial functions grow too tangled to integrate by hand — a wavefunction with an
explicit $r_{12}$ correlation factor — we evaluate $\langle H\rangle$ by **variational Monte Carlo**. We
write it as an average of the **local energy** $E_L=H\psi/\psi$ over the probability distribution
$|\psi|^2$, sample configurations from $|\psi|^2$ with the **Metropolis algorithm** of Volume V ([§5.2](../05-classical-stat-mech/probability.ipynb),
[§5.8](../05-classical-stat-mech/partition-function.ipynb)), and average. The mean is the variational energy; its **variance** measures how far $\psi$ is from an
eigenstate — and *vanishes exactly at the true ground state*. This **zero-variance principle** is the
conceptual heart of the method and the foundation of modern quantum Monte Carlo, the same
sample-a-distribution-and-average-an-estimator logic that runs through computational many-body physics
(and, in a different guise, through the sampling behind machine-learned interatomic potentials).

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the trial energies via `numpy.trapezoid` or analytic formulas,
`scipy.optimize.minimize_scalar` for the parameter optimization, the Metropolis sampler built on
`numpy.random.default_rng`, and the local energy $E_L=H\psi/\psi$ coded explicitly.

> **Conventions and method notes.** Atomic units ($\hbar=m_e=e=4\pi\varepsilon_0=1$), energies in Hartree
> ($1\,\text{Ha}=27.211\,$eV). Trial forms: Gaussian $e^{-\alpha x^2/2}$ (oscillator), exponential
> $e^{-\alpha r}$ (hydrogen), the product $e^{-Z(r_1+r_2)}$ and its Jastrow-correlated cousin (helium).
> The VMC uses **single-electron** Metropolis moves, a **burn-in** before averaging, a step tuned for an
> acceptance near $0.5$, and a **block-averaged** error bar (honest about the autocorrelation of Metropolis
> samples — a naive $\sigma/\sqrt N$ underestimates it). See Griffiths and Sakurai & Napolitano (the
> variational principle, helium); Thijssen, *Computational Physics*, and Foulkes et al. (Rev. Mod. Phys.
> 2001) for VMC; and Notebooks [§6.21](perturbation-fine-structure.ipynb) (perturbation theory — the complementary method), [§6.17](hydrogen-atom.ipynb) (hydrogenic
> orbitals), [§6.12](harmonic-oscillator.ipynb) (the oscillator), [§5.2](../05-classical-stat-mech/probability.ipynb)/[§5.8](../05-classical-stat-mech/partition-function.ipynb) (Monte Carlo, the Metropolis algorithm).

## Theory in brief

### The variational principle

For any normalized trial state $|\psi\rangle$,

```{math}
:label: eq-variational
\langle\psi|H|\psi\rangle\ge E_0 ,
```

with equality iff $|\psi\rangle$ is the ground state. **Proof:** expand $|\psi\rangle=\sum_n c_n|n\rangle$
in the (unknown) energy eigenbasis; then $\langle H\rangle=\sum_n|c_n|^2 E_n\ge E_0\sum_n|c_n|^2=E_0$,
since every $E_n\ge E_0$. The bound is **one-sided** — the estimate is always *above* the truth, so lower
is better. (Unnormalized: use $\langle\psi|H|\psi\rangle/\langle\psi|\psi\rangle$.)

### The method, and the warm-ups

The bound {eq}`eq-variational` holds for *every* trial state, so it holds in particular for the best
member of any parametrized family $\psi_\alpha$, and that observation is already the whole algorithm:

```{math}
:label: eq-trial
E(\alpha)=\langle\psi_\alpha|H|\psi_\alpha\rangle,\qquad \min_\alpha E(\alpha)\ge E_0 .
```

Choose a family $\psi_\alpha$, compute $E(\alpha)$, minimize. A Gaussian $e^{-\alpha x^2/2}$ for the
oscillator gives $E(\alpha)=\alpha/4+1/4\alpha$, minimized at $\alpha=1\Rightarrow E=\tfrac12=E_0$; an
exponential $e^{-\alpha r}$ for hydrogen gives $E(\alpha)=\alpha^2/2-\alpha$, minimized at $\alpha=1
\Rightarrow-\tfrac12\,$Ha. Both families *contain* the exact ground state, so the bound is **saturated**.

### Helium and screening

Helium is the method's first real test: two electrons bound to a $Z=2$ nucleus, each repelled by the
other. Its Hamiltonian, and the trial energy that a page of hydrogenic integrals produces from it
(Griffiths carries them out), are

```{math}
:label: eq-helium
H=-\tfrac12\nabla_1^2-\tfrac12\nabla_2^2-\frac{2}{r_1}-\frac{2}{r_2}+\frac{1}{r_{12}},\qquad E(Z)=Z^2-\tfrac{27}{8}Z\ \Rightarrow\ Z_{\text{eff}}=\tfrac{27}{16} .
```

The $1/r_{12}$ repulsion is $\sim37\%$ of the binding — not small, and non-separable. The trial
$\psi=e^{-Z(r_1+r_2)}$ with $Z$ variational gives $E(Z)=Z^2-\tfrac{27}{8}Z$ (kinetic $Z^2$, electron–
nucleus $-4Z$, electron–electron $+\tfrac58 Z$), minimized at $Z_{\text{eff}}=27/16\approx1.69<2$:
**screening**. The energy $-2.85\,$Ha beats the unscreened $Z=2$ value ($-2.75$) and nears experiment
($-2.90$); the residual gap is **correlation**, which the Monte Carlo below addresses.

### Variational Monte Carlo and the local energy

When the trial function grows too tangled for closed-form integrals, we evaluate $\langle H\rangle$
statistically instead. Multiplying and dividing the integrand of $\int\psi^* H\psi$ by $\psi$ recasts
the expectation value as

```{math}
:label: eq-vmc
\langle H\rangle=\frac{\int|\psi|^2 E_L}{\int|\psi|^2},\qquad E_L(R)=\frac{H\psi(R)}{\psi(R)} ,
```

an average of the **local energy** $E_L$ over $|\psi|^2$. Sample configurations $R$ from $|\psi|^2$ with
**Metropolis** ([§5.2](../05-classical-stat-mech/probability.ipynb)/[§5.8](../05-classical-stat-mech/partition-function.ipynb): propose a move, accept with probability $\min(1,|\psi_{\text{new}}|^2/
|\psi_{\text{old}}|^2)$) and average $E_L$; the statistical error is $\sim\sigma(E_L)/\sqrt N$. This
evaluates trial functions — like one with an explicit $r_{12}$ **Jastrow** correlation factor — that have
no closed-form integral.

### The zero-variance principle

The local energy is more than a computational device, and one substitution shows why: feed an exact
eigenstate into its definition {eq}`eq-vmc` and the wavefunction cancels,

```{math}
:label: eq-zero-variance
H\psi=E\psi\ \Rightarrow\ E_L=\frac{H\psi}{\psi}=E\ \text{(constant)},\quad \mathrm{Var}(E_L)=0 .
```

If $\psi$ is an **exact** eigenstate, $E_L$ is the *same everywhere* — its variance is zero. So
$\mathrm{Var}(E_L)$ measures how far $\psi$ is from an eigenstate, and minimizing it optimizes trial
functions. This **zero-variance principle** is the foundation of modern quantum Monte Carlo.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize_scalar

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED, BLUE = "#c1121f", "#1d4e89"

HARTREE_EV = 27.211386  # 1 Hartree in eV
HE_EXPERIMENT = (
    -2.9037
)  # helium ground state (non-relativistic, infinite nuclear mass), Ha


def variational_energy_oscillator(alpha):
    r"""The oscillator trial energy $E(\alpha)=\alpha/4+1/4\alpha$ for $\psi_\alpha=e^{-\alpha x^2/2}$ {eq}`eq-trial`."""
    return alpha / 4 + 1 / (4 * alpha)


def variational_energy_hydrogen(alpha):
    r"""The hydrogen trial energy $E(\alpha)=\alpha^2/2-\alpha$ for $\psi_\alpha=e^{-\alpha r}$ {eq}`eq-trial`."""
    return alpha**2 / 2 - alpha


def variational_energy_helium(Z):
    r"""The helium trial energy $E(Z)=Z^2-\tfrac{27}{8}Z$ for $\psi=e^{-Z(r_1+r_2)}$ {eq}`eq-helium`."""
    return Z**2 - (27 / 8) * Z


def optimize_trial(energy_of_param, bounds):
    """Minimize a trial energy over its parameter with ``scipy.optimize.minimize_scalar`` (bounded).

    Returns the minimizing parameter and the minimum energy — the best (lowest, hence tightest) bound in
    the family {eq}`eq-trial`.
    """
    result = minimize_scalar(energy_of_param, bounds=bounds, method="bounded")
    return result.x, result.fun


def _jastrow(r12, b):
    r"""The Jastrow correlation exponent $u(r_{12})=\tfrac12 r_{12}/(1+b\,r_{12})$ (keeps the electrons apart)."""
    return 0.5 * r12 / (1 + b * r12)


def local_energy_helium(r1, r2, Z, b=0.0):
    r"""The helium local energy $E_L=H\psi/\psi$ for the trial $\psi=e^{-Z(r_1+r_2)+u(r_{12})}$ {eq}`eq-vmc`.

    For the plain product ($b=0$): $E_L=-Z^2+(Z-2)(1/r_1+1/r_2)+1/r_{12}$. A nonzero ``b`` adds the Jastrow
    factor $u=\tfrac12 r_{12}/(1+b r_{12})$ and its kinetic contribution (derived analytically; verified
    against a finite-difference Laplacian to a part in $10^7$). ``r1``, ``r2`` are the two 3-vectors.
    """
    n1 = np.linalg.norm(r1)
    n2 = np.linalg.norm(r2)
    d = r1 - r2
    n12 = np.linalg.norm(d)
    e_l = -(Z**2) + (Z - 2) * (1 / n1 + 1 / n2) + 1 / n12
    if b != 0.0:
        u_prime = 0.5 / (1 + b * n12) ** 2  # u'(r12)
        u_second = -b / (1 + b * n12) ** 3  # u''(r12)
        cross = (r1 / n1 - r2 / n2) @ (d / n12)
        e_l += -(u_prime**2) - u_second - 2 * u_prime / n12 + Z * u_prime * cross
    return e_l


def metropolis_vmc(Z, b=0.0, n_steps=120000, step=0.6, burn=8000, n_blocks=80, seed=1):
    r"""Variational Monte Carlo for helium: sample $|\psi|^2$ by Metropolis, average the local energy {eq}`eq-vmc`.

    Single-electron Metropolis moves (``numpy.random.default_rng``): propose $r_i\to r_i+\text{step}\cdot
    \mathcal N$, accept with probability $\min(1,|\psi_{\text{new}}|^2/|\psi_{\text{old}}|^2)$. After a
    burn-in, average ``local_energy_helium``. The error bar is **block-averaged** (honest about
    Metropolis autocorrelation, which a naive $\sigma/\sqrt N$ underestimates).

    Returns
    -------
    mean, error, spread, acceptance : float
        The VMC energy, its (block) statistical error, the local-energy standard deviation, and the
        Metropolis acceptance rate.
    """
    rng = np.random.default_rng(seed)
    r = [rng.standard_normal(3) / Z, rng.standard_normal(3) / Z]

    def log_psi(cfg):  # ln|ψ| = −Z(r1+r2) + u(r12)
        base = -Z * (np.linalg.norm(cfg[0]) + np.linalg.norm(cfg[1]))
        return base + (
            _jastrow(np.linalg.norm(cfg[0] - cfg[1]), b) if b != 0.0 else 0.0
        )

    s = log_psi(r)
    accepted = 0
    samples = []
    for k in range(n_steps):
        i = k % 2  # alternate which electron moves
        trial = [r[0].copy(), r[1].copy()]
        trial[i] = r[i] + step * rng.standard_normal(3)
        s_trial = log_psi(trial)
        if np.log(rng.random()) < 2 * (s_trial - s):  # |ψ|² ratio = exp(2Δ ln|ψ|)
            r, s = trial, s_trial
            accepted += 1
        if k >= burn:
            samples.append(local_energy_helium(r[0], r[1], Z, b))
    samples = np.array(samples)
    blocks = (
        samples[: len(samples) // n_blocks * n_blocks]
        .reshape(n_blocks, -1)
        .mean(axis=1)
    )
    return (
        samples.mean(),
        blocks.std() / np.sqrt(n_blocks),
        samples.std(),
        accepted / n_steps,
    )

## Exercise 1 — The variational principle

Prove and illustrate that $\langle\psi|H|\psi\rangle\ge E_0$ for *any* trial state, with equality
only at the ground state {eq}`eq-variational`.

1. For a solvable $H$ (here a small Hermitian matrix), the exact $E_0$ is the lowest eigenvalue
   (`numpy.linalg.eigvalsh`).
2. The proof: a trial $|\psi\rangle=\sum c_n|n\rangle$ gives $\langle H\rangle=\sum|c_n|^2 E_n\ge
   E_0$.
3. Draw many random normalized trial vectors and compute
   $\langle\psi|H|\psi\rangle/\langle\psi|\psi\rangle$.
4. Confirm every one is $\ge E_0$, and that the ground eigenvector attains equality. Frame: the
   estimate is always an upper bound — lower is better.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    principle_ok,
    "the variational principle bounds the ground-state energy from above: ⟨H⟩ ≥ E₀ for every trial state, with equality at the ground state",
)

## Exercise 2 — Optimizing a trial: the oscillator

Use a Gaussian trial $\psi_\alpha=e^{-\alpha x^2/2}$ to estimate the harmonic-oscillator
ground-state energy, and show the bound is **saturated** at $E_0=\tfrac12$ {eq}`eq-trial`.

1. The trial energy is $E(\alpha)=\alpha/4+1/4\alpha$ (the `variational_energy_oscillator`
   helper).
2. Minimize over $\alpha$ with `scipy.optimize.minimize_scalar` (the `optimize_trial` helper).
3. Confirm the minimum is $E_0=\tfrac12$ at $\alpha=1$.
4. Note the trial family *contains* the exact Gaussian ground state, so the bound is tight. Frame:
   when the family is right, the bound is exact.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    E_star,
    0.5,
    "the Gaussian trial saturates the oscillator ground-state energy (E₀=½ at α=1)",
    atol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Optimizing a trial: hydrogen

Estimate the hydrogen ground-state energy with an exponential trial $\psi_\alpha=e^{-\alpha r}$,
again saturating the bound at $-\tfrac12\,$Ha {eq}`eq-trial`.

1. The trial energy is $E(\alpha)=\alpha^2/2-\alpha$ (the `variational_energy_hydrogen` helper).
2. Minimize with `optimize_trial`.
3. Confirm $-\tfrac12\,$Ha at $\alpha=1$.
4. Note the trial at $\alpha=1$ *is* the exact $1s$ orbital ([§6.17](hydrogen-atom.ipynb)). Frame: hydrogen as a second
   saturated bound, warming up for helium.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    E_h,
    -0.5,
    "the exponential trial saturates the hydrogen ground-state energy (−0.5 Ha at α=1)",
    atol=1e-3,
)

## Exercise 4 — Helium and screening

Estimate the helium ground-state energy variationally with an effective nuclear charge, and show
one parameter captures the **screening** that perturbation theory misses {eq}`eq-helium`.

1. The helium Hamiltonian has the non-separable repulsion $1/r_{12}$; the trial is
   $\psi=e^{-Z(r_1+r_2)}$ with $Z$ variational.
2. Its energy is $E(Z)=Z^2-\tfrac{27}{8}Z$ (the `variational_energy_helium` helper).
3. Minimize $\Rightarrow Z_{\text{eff}}=27/16\approx1.69$.
4. Evaluate $E=-2.85\,$Ha; compare to experiment ($-2.90$) and the unscreened $Z=2$ ($-2.75$).
   Frame: each electron screens the nucleus, and one variational parameter finds it.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    np.array([Z_eff, E_helium]),
    np.array([27 / 16, -2.8477]),
    "helium's screened variational energy (Z_eff=27/16, E=−2.85 Ha) is close to experiment and beats the unscreened guess",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Variational Monte Carlo: the local energy

Reformulate $\langle H\rangle$ as an average of the **local energy** $E_L=H\psi/\psi$ over
$|\psi|^2$, implement a Metropolis sampler, and confirm it reproduces the analytic helium
variational energy {eq}`eq-vmc`.

1. Write $\langle H\rangle=\langle E_L\rangle_{|\psi|^2}$ with $E_L=H\psi/\psi$; for the product
   trial, $E_L=-Z^2+(Z-2)(1/r_1+1/r_2)+1/r_{12}$ (the `local_energy_helium` helper).
2. Sample electron configurations from $|\psi|^2$ with single-electron Metropolis moves
   (`numpy.random.default_rng`, accept with $\min(1,\text{ratio})$ — the `metropolis_vmc` helper).
3. Average $E_L$ with a block-averaged error bar.
4. Confirm it matches the analytic $E(Z_{\text{eff}})=-2.85\,$Ha. Frame: Monte Carlo evaluates the
   high-dimensional integral (the 5.x sampling spine, now on wavefunctions).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    vmc_ok,
    "VMC reproduces the analytic variational energy of the helium product trial (within a few statistical error bars)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — The zero-variance principle

Show the local energy is **constant** when the trial is an exact eigenstate, and that its variance
measures the trial's quality — the conceptual heart of quantum Monte Carlo {eq}`eq-zero-variance`.

1. For hydrogen $\psi_\alpha=e^{-\alpha r}$, the local energy is $E_L=-\alpha^2/2+(\alpha-1) /r$
   (from $H\psi/\psi$).
2. At $\alpha=1$ (the exact ground state), $E_L=-\tfrac12$ *everywhere* — zero variance.
3. At $\alpha\ne1$, $E_L$ scatters with $r$.
4. Conclude $\mathrm{Var}(E_L)$ gauges distance from an eigenstate and can be minimized to
   optimize trials. Frame: the foundation of quantum Monte Carlo (and, in spirit, of
   sampling-based optimization generally).

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    zero_var_std,
    0.0,
    "the local energy is constant at the exact ground state (α=1) — the zero-variance principle",
    atol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Practical VMC: tuning and error bars *(student)*

Run the helium VMC across Metropolis step sizes and sample counts, and characterize the acceptance
rate and the statistical error — the honest texture of a Monte Carlo calculation {eq}`eq-vmc`.

1. Run VMC at several step sizes and record the acceptance rate.
2. Identify the step giving an acceptance near $\sim0.5$ (too large rejects everything; too small
   explores slowly).
3. Show the error bar shrinks as $\sim1/\sqrt N$.
4. Note the burn-in discarded before averaging. Frame: step tuning, equilibration, and error bars
   are features of a stochastic calculation, not bugs.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    good_step and sqrtN_ok,
    "practical VMC requires step tuning (acceptance ~0.5), equilibration, and reports a statistical error bar that scales as 1/√N",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — A Jastrow factor beats the product trial *(student / stretch)*

Add an explicit electron–electron **Jastrow** correlation factor to the helium trial and show it
lowers the VMC energy toward experiment, with a smaller local-energy variance {eq}`eq-vmc`,
{eq}`eq-zero-variance`.

1. Multiply the product trial by a Jastrow factor $e^{u(r_{12})}$, $u=\tfrac12 r_{12}/(1+b
   r_{12})$ (electrons kept apart).
2. Run VMC with the Jastrow local energy (the `local_energy_helium` helper with $b>0$; its kinetic
   term was derived analytically and checked against finite differences).
3. Show the optimized energy is *lower* than the simple product trial and closer to $-2.90\,$Ha.
4. Show the local-energy variance is *smaller* — the Jastrow removes the electron-coalescence
   mismatch. Frame: better trials, better bounds — the path modern QMC takes.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    jastrow_ok,
    "explicit correlation improves the variational energy: the Jastrow trial lowers the VMC energy below the product trial toward experiment, with reduced variance",
)

## Exercise 9 — A guess with a guarantee

The variational method asks only for a plausible *shape* and returns something rare in approximation: a
rigorous bound, never below the truth, so that lowering the energy always means getting closer. We watched
it **saturate** for the oscillator and hydrogen, where the trial family contained the exact state; then
**screen** the nucleus to solve helium with a single parameter ($Z_{\text{eff}}=27/16$) where low-order
perturbation theory faltered on the electron–electron repulsion; and finally, when the best trial
functions grew too tangled to integrate, we handed the wavefunction to **Monte Carlo** and averaged the
local energy — whose *stillness* at the exact ground state (the zero-variance principle) is the idea the
whole method turns on.

**This exercise (synthesis).** No new computation: the method, and its honesty, are the result.
Perturbation theory ([§6.21](perturbation-fine-structure.ipynb)) and the variational method are the **two hands** of approximate quantum
mechanics — one expands around what you can solve, the other guesses the answer and bounds its error. The
variational method's quiet strength is exactly that honesty: it can be wrong, but never *optimistically*,
and Monte Carlo lets the guess be as elaborate as the physics demands. This VMC machinery — Metropolis
sampling of $|\psi|^2$, local-energy averaging, variance as a quality measure — is the foundation of
modern electronic-structure and many-body computation, and it is the same sample-and-average logic that
underlies stochastic methods far beyond it, from diffusion Monte Carlo to the sampling behind
machine-learned interatomic potentials. The next notebook ([§6.23](wkb-semiclassical.ipynb)) turns to the opposite regime — not a
good guess at the wavefunction, but the short-wavelength limit where quantum mechanics shades into the
classical action of Volume II: the **WKB approximation**.

## Notebook summary

The variational method — a guess with a guarantee — and variational Monte Carlo.

- **The variational principle** {eq}`eq-variational`: $\langle\psi|H|\psi\rangle\ge E_0$ for any trial
  (proven by expanding in the eigenbasis) — an upper bound, so lower is better.
- **Optimizing trials** {eq}`eq-trial`: minimize $E(\alpha)$ (`scipy.optimize.minimize_scalar`); the
  Gaussian oscillator and exponential hydrogen trials saturate at $\tfrac12$ and $-\tfrac12\,$Ha.
- **Helium and screening** {eq}`eq-helium`: $E(Z)=Z^2-\tfrac{27}{8}Z$ gives $Z_{\text{eff}}=27/16<2$ and
  $-2.85\,$Ha — screening captured by one parameter, where perturbation theory struggles.
- **Variational Monte Carlo** {eq}`eq-vmc`: $\langle H\rangle=\langle E_L\rangle_{|\psi|^2}$ with $E_L=
  H\psi/\psi$, by Metropolis sampling (`numpy.random.default_rng`, [§5.2](../05-classical-stat-mech/probability.ipynb)/[§5.8](../05-classical-stat-mech/partition-function.ipynb)) — reproduces the analytic
  energy, with block-averaged error bars, tuned steps ($\sim0.5$ acceptance), and $1/\sqrt N$ convergence.
- **The zero-variance principle** {eq}`eq-zero-variance`: $E_L$ is constant at an exact eigenstate; its
  variance gauges trial quality — the foundation of quantum Monte Carlo. A Jastrow factor lowers helium's
  energy toward experiment with reduced variance.

A plausible shape, a rigorous bound, and Monte Carlo to make the guess as rich as the physics needs.
Next: the semiclassical limit.

## Outlook

- **The WKB / semiclassical approximation ([§6.23](wkb-semiclassical.ipynb))**: the short-wavelength limit and the classical action —
  the connection to the Hamilton–Jacobi theory of Volume II ([§2.10](../02-classical-mechanics/hamilton-jacobi.ipynb)).
- **Time-dependent perturbation theory and Fermi's golden rule ([§6.24](time-dependent-perturbation.ipynb))**: transitions and spectra.
- **Quantum Monte Carlo beyond VMC**: diffusion Monte Carlo, the fermion sign problem, electronic-structure
  methods (horizons — and a nod to the sampling that powers modern computational materials science).
- **Correlation and the many-electron problem**: Hartree–Fock and beyond (horizons; Volume VII / quantum
  chemistry).
- **Cross-reference** [§6.21](perturbation-fine-structure.ipynb) (perturbation theory — the complementary method), [§6.17](hydrogen-atom.ipynb) (hydrogenic orbitals),
  [§6.12](harmonic-oscillator.ipynb) (the oscillator), [§5.2](../05-classical-stat-mech/probability.ipynb)/[§5.8](../05-classical-stat-mech/partition-function.ipynb) (Monte Carlo / Metropolis), and forward to [§6.23](wkb-semiclassical.ipynb), [§6.24](time-dependent-perturbation.ipynb).

In [ ]:
from ecp.style import footer

footer()